In [17]:
import glob
import pandas as pd
from pathlib import Path
from typing import Callable

files = glob.glob("/g/data/ct11/access-nri/replicas/esmvaltool/**/*.nc",recursive=True)
print(f"No. of obs6 files: {pd.Series(files).str.lower().str.contains('obs6').sum()}")
srs = pd.Series(files)
srs.name='path'

for f in files[:10]:
    print(f)

No. of obs6 files: 2886
/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier3/ERA-Interim/OBS6_ERA-Interim_reanaly_1_Emon_tdps_201701-201712.nc
/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier3/ERA-Interim/OBS6_ERA-Interim_reanaly_1_day_rsds_201101-201112.nc
/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier3/ERA-Interim/OBS6_ERA-Interim_reanaly_1_Amon_hur_199401-199412.nc
/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier3/ERA-Interim/OBS6_ERA-Interim_reanaly_1_Amon_rsutcs_199401-199412.nc
/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier3/ERA-Interim/OBS6_ERA-Interim_reanaly_1_Amon_vas_200801-200812.nc
/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier3/ERA-Interim/OBS6_ERA-Interim_reanaly_1_Amon_rsut_198101-198112.nc
/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier3/ERA-Interim/OBS6_ERA-Interim_reanaly_1_Amon_tauu_199201-199212.nc
/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier3/ERA-Interim/OBS6_ERA-Interi

In [84]:
def get_tier(file_path : str) -> str:
    """ Split the parts, find the one that is 
    either Tier1, Tier2, or Tier3.
    """
    path_parts = Path(file_path).parts
    for part in path_parts:
        if part in ['Tier1', 'Tier2', 'Tier3']:
            return part
    return None

def filter_for_str(srs: pd.Series, filter_str: str) -> pd.Series:
    """ Apply filtering function to each file path and expand results into separate columns """
    if not isinstance(srs, pd.Series):
        raise TypeError(f'Wrong dtype, probably the wrong input source. Expected {type(srs)}=pd.Series, got {type(srs)=}')
    
    return srs[
        srs.str.lower().str.contains(filter_str)
    ].reset_index(drop=True)

def split(filtered_srs: pd.Series, split_func: Callable) -> pd.DataFrame:
    """
    Take our filtered series & split out the various identifiers in the file stems using
    the `split_func` we pass in.
    """
    COLNAMES = ['project_id', 'source_id', 'experiment_id', 'version',  'table_id', 'variable_id', 'time_range', 'tier']
    df = pd.DataFrame(filtered_srs)
    df[['project_id', 'source_id', 'experiment_id', 'version', 'table_id', 'variable_id', 'time_range', 'tier']] = df['path'].apply(lambda x: pd.Series(split_func(x)))
    return df
            

In [85]:
from pathlib import Path
from typing import Callable

# Function to split the filename into components based on '_'
def split_filename_obs6(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.
    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range, *_]:
            return {
                'project_id': project_id,
                'source_id': source_id,
                'experiment_id': experiment_id,
                'version': version,
                'table_id': table_id,
                'variable_id': variable_id,
                'time_range': time_range,
                'tier' : get_tier(file_path),
            }
        case [project_id, source_id, experiment_id, version, table_id, variable_id]:
            return {
                'project_id': project_id,
                'source_id': source_id,
                'experiment_id': experiment_id,
                'version': version,
                'table_id': table_id,
                'variable_id': variable_id,
                'time_range': None,
                'tier' : get_tier(file_path),
            }  
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )
            


obs6_srs = filter_for_str(srs,'obs6')
obs6_df = split(obs6_srs, split_filename_obs6)


In [86]:
df_all = pd.DataFrame(pd.Series(files))
df_all.columns = ['path']
df_remaining = df_all[~pd.Series(files).str.lower().str.contains('obs6')].copy().reset_index(drop=True)

obs4mips_srs = filter_for_str(srs,'obs4mips')
obs4mips_df = pd.DataFrame(obs4mips_srs)


def split_filename_obs4mips(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.
    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    
    if "-" in components[-1]:
        time_range = components[-1]
    else:
        time_range = "-".join([components[-2], components[-1]])
    return {
        'project_id': "obs4MIPs",
        'source_id': components[2],
        'version': components[4],
        'variable_id': components[0],
        'time_range': time_range,
        'tier' : get_tier(file_path),
        } 
    
# Apply function to each file path and expand results into separate columns

obs4mips_df[['project_id', 'source_id', 'version', 'variable_id', 'time_range','tier']] = obs4mips_df['path'].apply(lambda x: pd.Series(split_filename_obs4mips(x)))
obs4mips_df

,path,project_id,source_id,version,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CALIOP,CALIPSO-GOCCP-v2.1,cllcalipso,20071201-20071231,Tier1
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CALIOP,CALIPSO-GOCCP-v2.1,clhcalipso,20100501-20100531,Tier1
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CALIOP,CALIPSO-GOCCP-v2.1,clmcalipso,20080801-20080831,Tier1
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CALIOP,CALIPSO-GOCCP-v2.1,clccalipso,20080701-20080731,Tier1
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CALIOP,CALIPSO-GOCCP-v2.1,cllcalipso,20090501-20090531,Tier1
...,...,...,...,...,...,...,...
1741,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,ISCCP,V1.0,albisccp,200402-200402,Tier1
1742,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,ISCCP,V1.0,albisccp,199504-199504,Tier1
1743,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,ISCCP,V1.0,albisccp,200001-200001,Tier1
1744,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,ISCCP,V1.0,cltisccp,198812-198812,Tier1


In [87]:
def split_filename_CFSR(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `zg_Amon_reanalysis_CFSR_201301-201312` which get mapped as
    f{variable_id}_{table_id}_{experiment_id}_{source_id}_{time_range}
    
    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [variable_id, table_id, experiment_id, source_id, time_range]:
            return {
                'project_id': "obs4MIPs",
                'source_id': source_id,
                'experiment_id': experiment_id,
                'version': None,
                'table_id': table_id,
                'variable_id': variable_id,
                'time_range': time_range,
                'tier' : get_tier(file_path),
            }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )


cfsr_srs = filter_for_str(srs,'cfsr')
cfsr_df = split(cfsr_srs, split_filename_CFSR)

cfsr_df

,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CFSR,reanalysis,None,Amon,zg,201301-201312,Tier1
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CFSR,reanalysis,None,Amon,va,200601-200612,Tier1
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CFSR,reanalysis,None,Amon,zg,198101-198112,Tier1
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CFSR,reanalysis,None,Amon,zg,200701-200712,Tier1
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CFSR,reanalysis,None,Amon,zg,198501-198512,Tier1
...,...,...,...,...,...,...,...,...,...
165,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CFSR,reanalysis,None,Amon,ua,201101-201112,Tier1
166,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CFSR,reanalysis,None,Amon,zg,200201-200212,Tier1
167,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CFSR,reanalysis,None,Amon,ua,199601-199612,Tier1
168,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CFSR,reanalysis,None,Amon,ua,199901-199912,Tier1


In [88]:
def split_filename_CERES_EBAF(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `rlds_CERES-EBAF_L3B_Ed2-7_200003-201209` which get mapped as
    f{variable_id}_{source_id}_{experiment_id}_{version_}_{time_range}
    or 
    `OBS_CERES-EBAF_sat_Ed4.0_Amon_rsut_200003-201812` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range}"

    
    
    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [variable_id, source_id, experiment_id, version, time_range]:
            return {
                'project_id': "obs4MIPs",
                'source_id': source_id,
                'experiment_id': experiment_id,
                'version': version,
                'table_id': None,
                'variable_id': variable_id,
                'time_range': time_range,
                'tier' : get_tier(file_path),
            }
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                'project_id': project_id,
                'source_id': source_id,
                'experiment_id': experiment_id,
                'version': version,
                'table_id': table_id,
                'variable_id': variable_id,
                'time_range': time_range,
                'tier' : get_tier(file_path),
            }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )


ceres_ebaf_srs = filter_for_str(srs,'ceres-ebaf')
ceres_ebaf_df = split(ceres_ebaf_srs, split_filename_CERES_EBAF)

ceres_ebaf_df

,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,CERES-EBAF,sat,Ed4.0,Amon,rsut,200003-201812,Tier2
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,CERES-EBAF,sat,Ed4.0,Amon,rlutcs,200003-201812,Tier2
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,CERES-EBAF,sat,Ed4.1,Amon,rlut,200003-202203,Tier2
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,CERES-EBAF,sat,Ed4.1,Amon,rsut,200003-202203,Tier2
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,CERES-EBAF,sat,Ed4.1,Amon,rsutcs,200003-202203,Tier2
5,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,CERES-EBAF,sat,Ed4.1,Amon,rlutcs,200003-202203,Tier2
6,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,CERES-EBAF,sat,Ed4.0,Amon,rsutcs,200003-201812,Tier2
7,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,CERES-EBAF,sat,Ed4.0,Amon,rlut,200003-201812,Tier2
8,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CERES-EBAF,L3B,Ed2-8,None,rsdscs,200003-201311,Tier1
9,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,obs4MIPs,CERES-EBAF,L3B,Ed2-8,None,rlus,200003-201311,Tier1


In [89]:
# 
def split_filename_JRA55(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `rsut_Amon_reanalysis_JRA-55_195801-201912` which get mapped as
    f{variable_id}_{table_id}_{experiment_id}_{source_id}_{time_range}
    or 
    `OBS6_JRA-55_reanaly_1_Amon_tas_195801-202212` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range}"

    
    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [variable_id, table_id, experiment_id, source_id, time_range]:
            return {
                'project_id': "obs4MIPs",
                'source_id': source_id,
                'experiment_id': experiment_id,
                'version': None,
                'table_id': table_id,
                'variable_id': variable_id,
                'time_range': time_range,
                'tier' : get_tier(file_path),
            }
        case [_, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                'project_id': "OBS6",
                'source_id': source_id,
                'experiment_id': experiment_id,
                'version': version,
                'table_id': table_id,
                'variable_id': variable_id,
                'time_range': time_range,
                'tier' : get_tier(file_path),
            }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

jra55_srs = filter_for_str(srs,'jra-55')
jra55_df = split(jra55_srs, split_filename_JRA55)


jra55_flist = jra55_df['path'].tolist()
obs6_flist = obs6_df['path'].tolist()

### For whatever reason about half of these overlap? IDK, We drop the duplicates later on 
print(len(set(jra55_flist).intersection(set(obs6_flist))))
print(len(jra55_df))

16
29


In [98]:
#  '/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/ESACCI-SST/OBS_ESACCI-SST_sat_2.2_Amon_ts_201901-201912.nc',
# '/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/ESACCI-SST/OBS_ESACCI-SST_sat_L4-GHRSST-SSTdepth-OSTIA-GLOB_Amon_tsStderr_199201-199201.nc'

def split_filename_ESACCI(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_ESACCI-SST_sat_2.2_Amon_ts_201901-201912` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range}"
    or 
    `OBS_ESACCI-SST_sat_L4-GHRSST-SSTdepth-OSTIA-GLOB_Amon_tsStderr_199201-199201` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{grid_label}_{table_id}_{variable_id}_{time_range}"
    or
    `xco2_ghgcci_l3_v100_200301_201412.nc`, which get mapped as
    f"{variable_id`_{source_id}_{grid_label}_{version}_{t1}_{t2}" => f"{variable_id`_{}_{}_{version}_{time_range}"
    
    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id': table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case [variable_id, source_id, version, _, t_start, t_end]:
            return {
                'project_id': "Obs4MIPs",
                'source_id': source_id,
                'experiment_id': None,
                'version': version,
                'table_id': None,
                'variable_id': variable_id,
                'time_range': f"{t_start}-{t_end}",
                'tier' : get_tier(file_path),
            }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

esacci_srs = filter_for_str(srs,'esacci')
esacci_df = split(esacci_srs, split_filename_ESACCI)

esacci_df

,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESACCI-SOILMOISTURE,sat,L3S-SSMV-COMBINED-v4.2,Lmon,sm,197901-201612,Tier2
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESACCI-SOILMOISTURE,sat,L3S-SSMV-COMBINED-v4.2,Lmon,smStderr,197901-201612,Tier2
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESACCI-SOILMOISTURE,sat,L3S-SSMV-COMBINED-v4.2,Lmon,dosStderr,197901-201612,Tier2
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESACCI-SOILMOISTURE,sat,L3S-SSMV-COMBINED-v4.2,Lmon,dos,197901-201612,Tier2
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS6,ESACCI-OC,sat,fv5.0,Omon,chl,199709-202012,Tier2
...,...,...,...,...,...,...,...,...,...
595,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESACCI-OZONE,sat,L3,Amon,tozStderr,199701-201012,Tier2
596,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESACCI-OZONE,sat,L3,Amon,toz,199701-201012,Tier2
597,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESACCI-OZONE,sat,L3,Amon,tro3prof,200701-200812,Tier2
598,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,Obs4MIPs,ghgcci,None,l3,None,xco2,200301-201412,Tier1


In [97]:
# '/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/ESRL/OBS_ESRL_ground_PAL_Amon_co2s_200112-201912.nc',

def split_filename_ESRL(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_ESRL_ground_PAL_Amon_co2s_200112-201912.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{_}_{table_id}_{variable_id}_{time_range}"
 
    
    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, _, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': None,
                    'table_id': table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

esrl_srs = filter_for_str(srs,'esrl')
esrl_df = split(esrl_srs, split_filename_ESRL)

esrl_df

,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESRL,ground,None,Amon,co2s,197903-199008,Tier2
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESRL,ground,None,Amon,co2s,201003-201912,Tier2
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESRL,ground,None,Amon,co2s,199710-200908,Tier2
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESRL,ground,None,Amon,co2s,198001-202008,Tier2
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESRL,ground,None,Amon,co2s,199901-201105,Tier2
...,...,...,...,...,...,...,...,...,...
89,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESRL,ground,None,Amon,co2s,200204-201705,Tier2
90,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESRL,ground,None,Amon,co2s,197908-201912,Tier2
91,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESRL,ground,None,Amon,co2s,198612-201707,Tier2
92,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,ESRL,ground,None,Amon,co2s,199701-201912,Tier2


In [102]:
#  '/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/NCEP/OBS_NCEP_reanaly_1_Amon_zg_201001-201012.nc',

def split_filename_NCEP(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_NCEP_reanaly_1_Amon_zg_201001-201012.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range}"
 
    
    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id': table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

ncep_srs = filter_for_str(srs,'ncep')
ncep_df = split(ncep_srs, split_filename_NCEP)

ncep_df

,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS6,NCEP-DOE-R2,reanaly,2,Amon,wap,197901-202409,Tier2
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS6,NCEP-DOE-R2,reanaly,2,Amon,hur,197901-202409,Tier2
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS6,NCEP-DOE-R2,reanaly,2,Amon,tauu,197901-202409,Tier2
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS6,NCEP-DOE-R2,reanaly,2,Amon,clt,197901-202409,Tier2
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS6,NCEP-DOE-R2,reanaly,2,Amon,prw,197901-202409,Tier2
...,...,...,...,...,...,...,...,...,...
1248,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,NCEP,reanaly,1,day,rlut,199501-199512,Tier2
1249,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,NCEP,reanaly,1,day,pr,198601-198612,Tier2
1250,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,NCEP,reanaly,1,Amon,zg,196501-196512,Tier2
1251,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,NCEP,reanaly,1,day,pr,195501-195512,Tier2


In [106]:
#  '/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier2/WFDE5/OBS_WFDE5_ground_v1.1-CRU_day_tas_201301-201312.nc',

def split_filename_WFDE(file_path : str) -> dict[str, str | None]:
    """
    Take a filename & split it's stem on occurences of `_`. This
    returns a dictionary which we can then use to create a new 
    dataframe with all the various fields.

    These are of the form:
    `OBS_WFDE5_ground_v1.1-CRU_day_tas_201301-201312.nc` which get mapped as
    f"{project_id}_{source_id}_{experiment_id}_{version}_{table_id}_{variable_id}_{time_range}"
 
    
    """
    # Remove the file extension and split the rest based on '_'
    components = Path(file_path).stem.split('_')
    # Return components in a dictionary format for DataFrame usage
    match components:
        case [project_id, source_id, experiment_id, version, table_id, variable_id, time_range]:
            return {
                    'project_id': project_id,
                    'source_id': source_id,
                    'experiment_id': experiment_id,
                    'version': version,
                    'table_id': table_id,
                    'variable_id': variable_id,
                    'time_range': time_range,
                    'tier' : get_tier(file_path),
                }
        case _:
            raise RuntimeError(
                f"Unable to match all rows: failed on path {file_path}"
            )

wfde_srs = filter_for_str(srs,'wfde')
wfde_df = split(wfde_srs, split_filename_WFDE)

wfde_df

,path,project_id,source_id,experiment_id,version,table_id,variable_id,time_range,tier
0,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,WFDE5,ground,v1.1-CRU+GPCC,day,pr,199401-199412,Tier2
1,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,WFDE5,ground,v1.1-CRU,Amon,tas,199401-199412,Tier2
2,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,WFDE5,ground,v1.1-CRU,Amon,tas,200401-200412,Tier2
3,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,WFDE5,ground,v1.1-CRU+GPCC,day,pr,199501-199512,Tier2
4,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,WFDE5,ground,v1.1-CRU,Amon,pr,198001-198012,Tier2
...,...,...,...,...,...,...,...,...,...
227,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,WFDE5,ground,v1.1-CRU,Amon,tas,200501-200512,Tier2
228,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,WFDE5,ground,v1.1-CRU,day,tas,198601-198612,Tier2
229,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,WFDE5,ground,v1.1-CRU,Amon,pr,200401-200412,Tier2
230,/g/data/ct11/access-nri/replicas/esmvaltool/ob...,OBS,WFDE5,ground,v1.1-CRU,Amon,pr,201301-201312,Tier2


In [107]:
merged_df_1 = pd.concat([obs6_df, obs4mips_df, cfsr_df, ceres_ebaf_df, jra55_df, esacci_df, esrl_df, ncep_df, wfde_df]).reset_index(drop=True).drop_duplicates()

In [108]:
nfiles_todo = len(srs) - len(merged_df_1)
print(f"Remaining files: {nfiles_todo}")

Remaining files: 550


In [109]:
files_to_do = set(srs) - set(merged_df_1.path)
sorted(files_to_do)

['/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier1/AIRS-2-0/hur_AIRS-2-0_L3_v2_200209-201105.nc',
 '/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier1/AIRS-2-1/hur_AIRS-2-1_BE_gn_200209-201609.nc',
 '/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier1/AIRS-2-1/hur_mon_AIRS-2-1_BE_gn_200209-201609.nc',
 '/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier1/AIRS-2-1/hus_AIRS-2-1_BE_gn_200209-201609.nc',
 '/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier1/AIRS-2-1/hus_mon_AIRS-2-1_BE_gn_200209-201609.nc',
 '/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier1/AIRS-2-1/ta_AIRS-2-1_BE_gn_200209-201609.nc',
 '/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier1/AIRS-2-1/ta_mon_AIRS-2-1_BE_gn_200209-201609.nc',
 '/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier1/AIRS/husNobs_AIRS_L3_RetStd-v5_200209-201105.nc',
 '/g/data/ct11/access-nri/replicas/esmvaltool/obsdata-v2/Tier1/AIRS/husStderr_AIRS_L3_RetStd-v5_200209-2011